In [5]:
from collections import deque
import copy

easy = """004030050
609400000
005100489
000060930
300807002
026040000
453009600
000004705
090050200"""

medium = """530070000
600195000
098000060
800060003
400803001
700020006
060000280
000419005
000080079"""

hard = """102040007
000080000
009500304
000607900
540000026
006405000
708003400
000010000
200060509"""

veryhard = """001007000
600400300
000030064
380076000
000000036
270015000
000020051
700100200
008009000"""

boards = {
    "easy.txt": easy,
    "medium.txt": medium,
    "hard.txt": hard,
    "veryhard.txt": veryhard
}

for filename, data in boards.items():
    with open(filename, "w") as f:
        f.write(data)



class SudokuCSP:
    def __init__(self, board):
        self.board = board
        self.variables = [(r, c) for r in range(9) for c in range(9)]
        self.domains = {}
        self.backtrack_calls = 0
        self.failures = 0
        self.initialize_domains()

    def initialize_domains(self):
        for r in range(9):
            for c in range(9):
                if self.board[r][c] != 0:
                    self.domains[(r, c)] = {self.board[r][c]}
                else:
                    self.domains[(r, c)] = set(range(1, 10))

    def get_neighbors(self, row, col):
        neighbors = set()

        for i in range(9):
            if i != col:
                neighbors.add((row, i))
            if i != row:
                neighbors.add((i, col))

        start_row = (row // 3) * 3
        start_col = (col // 3) * 3

        for r in range(start_row, start_row + 3):
            for c in range(start_col, start_col + 3):
                if (r, c) != (row, col):
                    neighbors.add((r, c))

        return neighbors

    def revise(self, xi, xj):
        revised = False
        to_remove = set()

        for x in self.domains[xi]:
            possible = False

            for y in self.domains[xj]:
                if x != y:
                    possible = True
                    break

            if not possible:
                to_remove.add(x)

        if len(to_remove) > 0:
            self.domains[xi] = self.domains[xi] - to_remove
            revised = True

        return revised

    def ac3(self):
        queue = deque()

        for var in self.variables:
            neighbors = self.get_neighbors(var[0], var[1])

            for neighbor in neighbors:
                queue.append((var, neighbor))

        while len(queue) > 0:
            xi, xj = queue.popleft()

            if self.revise(xi, xj):
                if len(self.domains[xi]) == 0:
                    return False

                neighbors = self.get_neighbors(xi[0], xi[1])

                for neighbor in neighbors:
                    if neighbor != xj:
                        queue.append((neighbor, xi))

        return True

    def is_consistent(self, var, value):
        neighbors = self.get_neighbors(var[0], var[1])

        for neighbor in neighbors:
            if len(self.domains[neighbor]) == 1:
                neighbor_value = next(iter(self.domains[neighbor]))

                if neighbor_value == value:
                    return False

        return True

    def forward_check(self, var):
        queue = deque()
        queue.append(var)

        while len(queue) > 0:
            current = queue.popleft()

            if len(self.domains[current]) != 1:
                continue

            value = next(iter(self.domains[current]))

            neighbors = self.get_neighbors(current[0], current[1])

            for neighbor in neighbors:
                if value in self.domains[neighbor]:
                    if len(self.domains[neighbor]) == 1:
                        return False

                    self.domains[neighbor].remove(value)

                    if len(self.domains[neighbor]) == 0:
                        return False

                    if len(self.domains[neighbor]) == 1:
                        queue.append(neighbor)

        return True

    def is_complete(self):
        for var in self.variables:
            if len(self.domains[var]) != 1:
                return False

        return True

    def select_unassigned_variable(self):
        unassigned = []

        for var in self.variables:
            if len(self.domains[var]) > 1:
                unassigned.append(var)

        return min(unassigned, key=lambda x: len(self.domains[x]))

    def backtrack(self):
        self.backtrack_calls += 1

        if self.is_complete():
            return True

        var = self.select_unassigned_variable()

        values = sorted(self.domains[var])

        for value in values:
            if self.is_consistent(var, value):
                saved_domains = copy.deepcopy(self.domains)

                self.domains[var] = {value}

                if self.forward_check(var):
                    if self.backtrack():
                        return True

                self.domains = saved_domains

        self.failures += 1
        return False

    def solve(self):
        if not self.ac3():
            return False

        return self.backtrack()

    def print_solution(self):
        for r in range(9):
            row = ""

            for c in range(9):
                row += str(next(iter(self.domains[(r, c)])))

            print(row)



def read_board(filename):
    board = []

    with open(filename, "r") as file:
        for line in file:
            row = []

            for ch in line.strip():
                row.append(int(ch))

            board.append(row)

    return board



files = ["easy.txt", "medium.txt", "hard.txt", "veryhard.txt"]

for filename in files:
    print("\n")
    print("Solving:", filename)

    board = read_board(filename)

    solver = SudokuCSP(board)

    if solver.solve():
        print("Solved Sudoku:")
        solver.print_solution()
        print("Backtrack Calls:", solver.backtrack_calls)
        print("Failures:", solver.failures)
    else:
        print("No solution exists.")



Solving: easy.txt
Solved Sudoku:
784932156
619485327
235176489
578261934
341897562
926543871
453729618
862314795
197658243
Backtrack Calls: 1
Failures: 0


Solving: medium.txt
Solved Sudoku:
534678912
672195348
198342567
859761423
426853791
713924856
961537284
287419635
345286179
Backtrack Calls: 1
Failures: 0


Solving: hard.txt
Solved Sudoku:
152346897
437189652
689572314
821637945
543891726
976425183
798253461
365914278
214768539
Backtrack Calls: 7
Failures: 2


Solving: veryhard.txt
Solved Sudoku:
431867925
652491387
897532164
384976512
519284736
276315849
943728651
765143298
128659473
Backtrack Calls: 56
Failures: 43
